In [ ]:
import pandas as pd
model_df=pd.read_csv("../data/processed/leishmania_ml_dataset.csv")
model_df.head()

,Molecule ChEMBL ID,SMILES,pIC50,Molecular_Weight,AlogP,TPSA,HBA,HBD,RO5_Violations,Rotatable_Bonds,QED,Aromatic_Rings,Heavy_Atoms,NP_Likeness
0,CHEMBL1000,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,3.940555,388.90,3.15,53.01,4.0,1.0,0.0,8.0,0.70,2.0,27.0,-1.21
1,CHEMBL100210,CCCC[C@H]1C(=O)O[C@@H]2O[C@@]3(CC)CC[C@H]4[C@H...,4.443697,338.44,3.96,53.99,5.0,0.0,0.0,4.0,0.57,0.0,24.0,2.52
2,CHEMBL100740,C[C@@H]1CC[C@H]2[C@@H](C)C(=O)O[C@@H]3OC(C)(C)...,3.800519,284.35,2.64,53.99,5.0,0.0,0.0,0.0,0.51,0.0,20.0,2.66
3,CHEMBL104,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,5.242633,344.85,5.38,17.82,2.0,0.0,1.0,4.0,0.45,4.0,25.0,-0.58
4,CHEMBL106,OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F,3.898138,306.28,0.74,81.65,7.0,1.0,0.0,5.0,0.75,3.0,22.0,-0.86


In [3]:
import rdkit
print(rdkit.__version__)

2026.03.3


## Generation of Molecular Objects

SMILES strings were converted into RDKit molecular objects to enable the calculation of molecular fingerprints and other structure-based representations required for machine learning.

In [4]:
from rdkit import Chem

model_df["Mol"] = model_df["SMILES"].apply(
    Chem.MolFromSmiles
)

model_df["Mol"].head()

0    <rdkit.Chem.rdchem.Mol object at 0x00000212FA6...
1    <rdkit.Chem.rdchem.Mol object at 0x00000212FA6...
2    <rdkit.Chem.rdchem.Mol object at 0x00000212FA6...
3    <rdkit.Chem.rdchem.Mol object at 0x00000212FA6...
4    <rdkit.Chem.rdchem.Mol object at 0x00000212FA6...
Name: Mol, dtype: object

In [5]:
model_df["Mol"].isnull().sum()

np.int64(0)

## Generation of Morgan Fingerprints

Morgan fingerprints were generated from molecular structures to capture structural patterns and chemical substructures. These fingerprints provide a high-dimensional representation of molecular features commonly used in QSAR modeling.

In [6]:
from rdkit.Chem import AllChem

fp = AllChem.GetMorganFingerprintAsBitVect(
    model_df["Mol"].iloc[0],
    radius=2,
    nBits=1024
)

len(fp)

[00:05:01] DEPRECATION WARNING: please use MorganGenerator


1024

In [7]:
import numpy as np

fingerprints = []

for mol in model_df["Mol"]:
    
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=1024
    )
    
    fingerprints.append(
        np.array(fp)
    )

[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerator
[00:06:10] DEPRECATION WARNING: please use MorganGenerat

In [8]:
fp_df = pd.DataFrame(
    fingerprints
)

fp_df.shape

(5457, 1024)

In [ ]:
fp_df.to_csv(
    "../data/processed/morgan_fingerprints.csv",
    index=False
)

## Construction of Feature Sets

Three feature representations were prepared for model development: molecular descriptors alone, Morgan fingerprints alone, and a combined descriptor-fingerprint representation. This approach allows comparison of different molecular representations for predicting anti-leishmanial activity.

In [14]:
descriptor_cols = [
    "Molecular_Weight",
    "AlogP",
    "TPSA",
    "HBA",
    "HBD",
    "RO5_Violations",
    "Rotatable_Bonds",
    "QED",
    "Aromatic_Rings",
    "Heavy_Atoms",
    "NP_Likeness"
]

X_desc = model_df[
    descriptor_cols
]

y = model_df["pIC50"]

print(X_desc.shape)

(5457, 11)


In [15]:
X_fp = fp_df.copy()

print(X_fp.shape)

(5457, 1024)


In [16]:
X_combined = pd.concat(
    [
        X_desc.reset_index(drop=True),
        X_fp.reset_index(drop=True)
    ],
    axis=1
)

print(X_combined.shape)

(5457, 1035)


In [ ]:
X_desc.to_csv(
    "../data/processed/X_descriptors.csv",
    index=False
)
X_fp.to_csv(
    "../data/processed/X_fingerprints.csv",
    index=False
)
X_combined.to_csv(
    "../data/processed/X_combined.csv",
    index=False
)
y.to_csv(
    "../data/processed/y_pIC50.csv",
    index=False
)

## Export of Feature Matrices

The descriptor, fingerprint, and combined feature matrices were exported for subsequent machine learning analyses. The target variable (pIC50) was also saved separately to facilitate reproducible model development and evaluation.